# Reproducing Tingley & Buzsáki (2018) from dandiset 000213

**Paper.** Tingley D, Buzsáki G (2018). Transformation of a spatial map across the
hippocampal-lateral septal circuit. *Neuron* 98, 1229-1242.
[doi:10.1016/j.neuron.2018.04.028](https://doi.org/10.1016/j.neuron.2018.04.028). This attempt used
the full text of the NIH author manuscript in PubMed Central (PMC6605060): Results, figure legends
and STAR Methods. The supplementary figures were not read.

**Dandiset.** [DANDI:000213](https://dandiarchive.org/dandiset/000213/0.250624.0438), version
0.250624.0438. The paper cites no version (its data statement says the data were being prepared for
CRCNS). There are two published versions, 0.220127.1738 and 0.250624.0438; they hold the same 67
files with the same asset identifiers and sizes (checked below) and differ only in dandiset-level
metadata, so the most recent is pinned.

**What the dandiset holds.** 67 sessions from the five rats (DT2, DT5, DT7, DT8, DT9): sorted units
labelled by region (`ls`, `hpc`, `ca3` and a few others), LFP at 1250 Hz, head position, and trials.
The paper's processed files (linearized behaviour, trial conditions, the CA1 channel of each session)
are not in it; they are rebuilt here as described in `lshpc.py`.

**Code read.** The paper's figure scripts, https://github.com/DavidTingley/papers (LS_phasecoding,
commit 9f4cd48a44f211956954558b66d776ccbb74d693); buzcode, https://github.com/buzsakilab/buzcode
(commit e615663b2b100782d1f378580f6d7aaa35c12937); and the per-cell decoder in
https://github.com/DavidTingley/scratch (commit 2f804532290774f52a442bf0f3ec23dd5398d58d). The
functions that bin phases (bz_phaseMap2Bins), compute information (olypherInfo) and detect
assemblies are not public.

**Claims.** The claims file is `claims/000213/10.1016-j.neuron.2018.04.028.yaml`.

| Claim | Tested here |
|---|---|
| ls-neuron-count: 1,647 LS neurons, 62 sessions | Tested |
| hpc-unit-count: 2,297 CA1 and 654 CA3 units | Tested |
| ls-active-neurons: 840 LS neurons fire at least 1.5 spikes per trial | Tested |
| fig2-ls-phase-vs-shuffle: 89% of LS neurons code position in phase | Tested |
| fig2-ls-phase-beats-rate: phase beats rate in 786/840 neurons, 3,046/4,393 conditions, 62/62 sessions | Tested |
| fig2e-ls-phase-lower-mse: phase MSE below rate MSE across LS conditions and sessions | Tested |
| fig2fg-mse-difference: mean phase minus rate MSE, LS and hippocampus | Tested |
| fig1i-hpc-rate-phase-equivalent: rate and phase decoding equivalent for place fields | Tested |
| ls-few-place-fields: 5.3% of LS and 31.7% of hippocampal rate maps have place fields | Tested |
| hpc-place-field-properties: 711 of 1,425 neurons with fields, 8.5 Hz, 38 bins | Tested |
| ls-place-fields-unstable: LS fields less stable (41.5 against 10) | Tested |
| fig1e-phase-info-smaller-scale: peak information at 15 (phase) and 29 (rate) bins | Tested |
| fig1g-phase-more-info-fine-scale: phase more informative below 6 cm, rate above 10 cm | Tested |
| fig4d-depth-gradient: phase coding grows with LS depth | Tested |
| fig4e-onset-depth: onset phase correlates with depth | Tested |
| fig6c-ca3-coherence-increases: LS-CA3 theta coherence rises along the run | Tested |
| fig6d-cfc-stronger-ca3: theta-gamma coupling rises, stronger for CA3 | Tested |
| figs7-ensemble-ls-phase-best: LS ensemble phases beat hippocampal ensembles | Tested |
| fig5-assemblies: 18.5% of pairs form gamma assemblies | Tested on a random sample of 1% of the pairs |
| ls-information-lower: 0.4 against 1.7 bits/cm | Tested |
| fig5g-assembly-neurons-stronger-phase | Blocked: not runnable (needs the assemblies of every pair) |
| fig6e-assembly-position | Blocked: not runnable (needs the assemblies of every pair) |
| speed-independent: 2.8% of conditions speed-related | Blocked: methods underspecified |
| figs6-reference-frame: maze rotation | Blocked: metadata insufficient |
| fig7-dynamic-weighting-model: spiking model | Blocked: other (a simulation) |

**Revision after the first verification round.** Place fields now use the Gaussian rate map that the
paper's code passes to bz_getPlaceFields1D (SD 4 bins), where the first version used the 5-bin box
map; Figure 6 groups sessions by their unit labels, as its criterion states, and leaves out sessions
without hippocampal units; the ensemble tests are one-sided; the theta reference of each session is
described and the phase claims are repeated on sessions with a clear theta rhythm; fig5-assemblies
is tested on a sample of pairs and ls-information-lower is tested, with criteria added to the claims
file in this revision.

**Second revision.** The sentences the second verifier found unsupported are corrected, and three
variants are added: place fields without the rule dropping peaks whose SD across trials exceeds
their mean, the onset-phase test on sessions with one theta reference label per rat, and the peer
prediction with the LS unit's spatial tuning as a covariate. No outcome changed.

The whole notebook streams the data and takes about twelve to fifteen minutes on four processes,
most of it the per-cell decoding and the peer prediction.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import lshpc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
results = lshpc.run_all(workers=4)
print(f"streamed and analysed {len(results)} files of dandiset {lshpc.DANDISET_ID} "
      f"version {lshpc.DANDISET_VERSION} in {time.time() - started:.0f} s")
errors = [(r["summary"]["path"], r["error"]) for r in results if "error" in r]
print("files that failed:", errors if errors else "none")

S = pd.DataFrame([r["summary"] for r in results])
S["session"] = S.index
S["name"] = S.session_id.str.slice(0, 32)
C = pd.DataFrame([dict(c, session=i, subject=results[i]["summary"]["subject"])
                  for i, r in enumerate(results) for c in r.get("cells", [])])
C["neuron"] = list(zip(C.session, C.unit))
A = C[C.active].copy()                 # included cell and condition pairs
LS = A[A.region == "ls"]
HP = A[A.region == "hpc"]
LSd = LS[LS.mse_phase.notna()]          # decoded (session has a theta channel)
HPd = HP[HP.mse_phase.notna()]

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
from dandi.dandiapi import DandiAPIClient

with DandiAPIClient() as client:
    ds = client.get_dandiset(lshpc.DANDISET_ID, lshpc.DANDISET_VERSION)
    meta = ds.get_raw_metadata()
    versions = [v.identifier for v in client.get_dandiset(lshpc.DANDISET_ID).get_versions()]
    listing = {}
    for v in versions:
        if v == "draft":
            continue
        listing[v] = {a.path: (a.identifier, a.size) for a in client.get_dandiset(lshpc.DANDISET_ID, v).get_assets()}
summary = meta["assetsSummary"]
print("Published versions:", [v for v in versions if v != "draft"])
pub = sorted(listing)
same = all(listing[pub[0]] == listing[v] for v in pub[1:])
print(f"Assets identical (path, asset id, size) across published versions: {same}; files: {len(listing[pub[0]])}")
print("Paper named by the dandiset:", [r.get("identifier") for r in meta.get("relatedResource", [])])
print("assetsSummary: subjects", summary.get("numberOfSubjects"), "| species", [s["name"] for s in summary.get("species", [])],
      "| variables", summary.get("variableMeasured"))
print("Subjects in the files:", sorted(S.subject.unique()), "| files per subject:", S.subject.value_counts().sort_index().to_dict())

The two published versions (2022 and 2025) hold the same 67 files with the same asset identifiers
and sizes; they differ only in dandiset-level metadata (contributor identifiers and affiliations,
the citation). The newest is pinned. The archive's summary of five rats of one species matches the
files. The paper's Methods state five rats, 3 male and 2 female; the files give no sex, so that
cannot be checked.

In [ ]:
units = (pd.DataFrame([dict(subject=s, location=l, n=n) for s, d in zip(S.subject, S.unit_locations) for l, n in d.items()])
         .pivot_table(index="subject", columns="location", values="n", aggfunc="sum", fill_value=0))
units.loc["all"] = units.sum()
print("Units by region label and subject:")
print(units.to_string())
sess = pd.DataFrame([dict(subject=s, location=l) for s, d in zip(S.subject, S.unit_locations) for l in d])
print("\nSessions holding units of each label:", sess.location.value_counts().to_dict())
print("Files without units:", S[S.n_units == 0].session_id.tolist())
print("Units with spike times not in increasing order:", int(S.units_unsorted.sum()),
      "in", S[S.units_unsorted > 0].session_id.tolist())
print("Units with spikes after the end of the LFP:", int(S.spikes_after_lfp_end.sum()))
print("\nElectrode labels per subject:")
for subj, g in S.groupby("subject"):
    labs = {}
    for d in g.electrode_locations:
        for k, v in d.items():
            labs[k] = labs.get(k, 0) + 1
    print(f"  {subj}: files with each label {labs}")
key = S[["n_spikes_total", "n_units"]].astype(str).agg("|".join, axis=1) + "|" + S.lfp_shape.astype(str)
print("\nFiles sharing total spikes, units and LFP shape:", S[key.duplicated(keep=False)].session_id.tolist())
print("Files sharing identical trial start times:", S[S.trial_starts_rounded.duplicated(keep=False)].session_id.tolist())
depth_rows = S[["subject", "session_id", "ls_depth", "hpc_depth", "has_hpc_units", "has_ca3_units"]]
print("\nDepths in the session names (um), first and second, with the hippocampal labels of the units:")
print(depth_rows.to_string(index=False))

Units carry a region label: `ls` (lateral septum), `hpc`, `ca3`, and a few cortical or other labels
(`acg`, `cc`, `ms`, `ppc`, `sFi`). One file (DT7, 20170317) has units without any label and is left
out of every count by region; one (DT7, 20170324) has no units. No two files share their spike
totals, unit counts and LFP length, or their trial times, so no recording was deposited twice.
Several sessions were recorded at the same depths on consecutive days (for example five DT8
sessions at 216 and 1944 um); each session's units are treated as separate units, as the paper's
counts do, and no claim assumes the same unit across sessions.

The electrode labels of sub-DT2 are all `unknown`; its hippocampal and LS channels are taken from
the electrode groups of its labelled units. The session names hold two depths that no metadata
describes. In DT9 the units are labelled `ca3` exactly in the sessions whose second depth is 828 um
or more, which fits the second depth being the hippocampal probe's; the first depth is taken as the
LS probe's. This is an assumption, used only by the depth claims. It is not settled by the other
rats: in DT5 the files labelled `hpc` and `ca3` alternate at second depths of 2232 to 2448 um, and the
DT8 depths (0 to 612 and 0 to 2088 um) do not say which probe is which.

In [ ]:
pos = S[["subject", "name", "task", "position_unit_stated", "position_extent_stated_units", "position_unit_used",
         "position_dt_median", "position_dt", "position_nonincreasing_steps", "position_z_equals_y",
         "position_nan_fraction", "path_length_median"]].copy()
pos["extent_1_99"] = pos.position_extent_stated_units.round(2)
print(pos.drop(columns="position_extent_stated_units").round(4).to_string(index=False))
odd = S[(S.position_unit_stated != "pixels") & (((S.position_unit_stated == "mm") & (S.position_extent_stated_units < 10)) |
                                               ((S.position_unit_stated == "m") & (S.position_extent_stated_units > 10)))]
print(f"\nFiles whose stated position unit disagrees with the values: {len(odd)}:", odd.name.tolist())
print("Files whose third position column equals the second:", int(S.position_z_equals_y.sum()), "of", len(S))
print("Files with repeated or decreasing position timestamps:", S[S.position_nonincreasing_steps > 0].name.tolist())

Position is stated in `m`, `mm` or `pixels`. Eight DT8 files state `mm` but hold values spanning about
1.1 to 1.3, which are metres (a maze of about a metre across; the same rat's other files span 1000
to 1200 mm). The unit used here is inferred from the extent: below 10 the values are taken as
metres, otherwise as millimetres; sub-DT2 positions stay in pixels, with no scale given. In every
file the third position column (z) is a copy of the second (y). In eight of the eleven sub-DT2 files
position timestamps repeat or go back (the count of such steps is printed), so the time per sample
is taken as the session's span over its sample count, which gives 33.3 ms (30 Hz) in every sub-DT2
file. The median path of a trial is 2.2 to 2.9 m on
the alternation mazes and 1.3 to 1.7 m on the linear track (outside sub-DT2), against the paper's
maze of about 3.2 m; the paper gives no other maze dimensions in the main text.

In [ ]:
tr = S[["subject", "name", "task", "n_trials", "trial_s_median", "trial_types", "directions", "conditions_used",
        "trials_without_positions", "trials_after_position_end"]]
print(tr.to_string(index=False))
print("\nTrial types:", sorted({str(t) for x in S.trial_types for t in x}), "| directions:", sorted({str(t) for x in S.directions for t in x}))
print("Files whose trials carry no type or direction:", S[S.trial_types.apply(lambda x: x == [""])].name.tolist())
print("Conditions with at least 10 trials, all files:", int(S.conditions_used.sum()))

The trials table has `trial_type` (central or wheel alternation) and `direction` (clockwise or
counter-clockwise) for the alternation tasks, and only start and stop times on the linear track and
in two DT8 alternation files (20170207 and 20170224).
Trials last a median of 1.6 to 5.2 s. The paper's trial conditions are not in the dandiset; the
conditions formed here (type, direction, start and end site) give 0 to 13 conditions of at least
10 trials per file. DT9 20170501 has none: its trials do not start or end at common sites (see the
clock check below).

In [ ]:
lf = S[["subject", "name", "lfp_shape", "lfp_chunks", "lfp_rate", "lfp_start", "lfp_conversion", "hpc_channels_from",
        "n_hpc_channels", "n_ls_channels", "theta_channel", "ls_channel"]].copy()
sel = [d if isinstance(d, dict) else {} for d in S.theta_channel_selection]
lf["theta_label"] = [d.get("label") for d in sel]
lf["theta_ratio"] = [d.get("theta_ratio", np.nan) for d in sel]
lf["median_ratio"] = [d.get("median_theta_ratio", np.nan) for d in sel]
lf["psd_peak_hz"] = [d.get("psd_peak_hz", np.nan) for d in sel]
lf["theta_contrast"] = [d.get("theta_peak_over_neighbours", np.nan) for d in sel]
lf["lfp_conversion"] = lf.lfp_conversion.map("{:.3g}".format)
S["theta_ratio"], S["theta_label"] = lf.theta_ratio.values, lf.theta_label.values
print(lf.round(2).to_string(index=False))
print("\nLFP sampling rates:", S.lfp_rate.unique(), "| starting times:", S.lfp_start.unique())
print("Files without a hippocampal channel (no theta reference):", S[~S.has_theta].name.tolist())
print("Hippocampal channel labels per file:", pd.Series([tuple(x) for x in S.hpc_channel_labels]).value_counts().to_dict(),
      "| files with both hpc and ca3 channels:", int(sum(("hpc" in x) and ("ca3" in x) for x in S.hpc_channel_labels)))
print("Label of the theta channel, files:", lf.theta_label.value_counts(dropna=False).to_dict())
q = lf[lf.theta_ratio.notna()].groupby("subject").agg(files=("name", "size"), ratio_below_1=("theta_ratio", lambda x: int((x < 1).sum())),
                                                       ratio_min=("theta_ratio", "min"), ratio_max=("theta_ratio", "max"),
                                                       psd_peak_min=("psd_peak_hz", "min"), psd_peak_max=("psd_peak_hz", "max"),
                                                       contrast_median=("theta_contrast", "median"))
print("\nTheta channel per subject: 6-9 Hz over 1-4 Hz power, frequency of the spectral peak between 2 and 20 Hz, and the")
print("6-9 Hz peak over the mean power at 2-5 and 11-15 Hz (theta_contrast):")
print(q.round(2).to_string())

The LFP is stored at 1250 Hz from time 0 in every file, as the paper's code assumes
(`phases(round(t*1250))`). 42 files chunk it as 500 samples by all channels, so the analysis reads
only the trials (with 1 s on each side), two channels at a time. One file (DT2, 20160217) has no
hippocampal units or labelled hippocampal channels, so it has no theta reference: its LS units are
counted, but not decoded.

No file has both `hpc` and `ca3` channels: the hippocampal channels of a file carry one label (43
files `hpc`, 11 `ca3`, and 12 `unknown`, taken from the groups of the `hpc` units). In the 11 files
with `ca3` channels the theta reference is therefore a CA3 channel. The paper's code does the same
for a session without a CA1 channel (it uses sessionInfo.ca1, else sessionInfo.ca3), but the paper's
Figure 6A counts 63 CA1 and 19 CA3 sessions, so its sessions had CA1 channels that are not labelled
here.

The theta reference is weak in DT5. On the chosen channel, 6 to 9 Hz power is below 1 to 4 Hz power
in 14 of 16 DT5 files (ratio 0.33 to 2.27), and the 6 to 9 Hz peak stands a median 3.5 times above
the power at 2 to 5 and 11 to 15 Hz, against medians of 11 to 97 in the other rats. The spectral
peak between 2 and 20 Hz is at 2 to 2.5 Hz in three DT5 files and one DT8 file, and at 7.25 to 9.75
Hz in all others. The phase claims are repeated on the sessions whose ratio is above 1.

In [ ]:
lags = list(lshpc.TRIAL_SHIFTS_S)
sp = pd.DataFrame([r["summary"]["clock_trial_speed"] for r in results], index=S.name)
sp_peak = sp.idxmax(axis=1)
print("Mean running speed (position units per s) inside the trial windows shifted by each lag (s); peak lag per file:")
print(pd.concat([sp.round(1), sp_peak.rename("peak")], axis=1).to_string())
print("\nPeak lag of speed inside trials, files per lag:", sp_peak.value_counts().sort_index().to_dict())
ratio = sp[0.0] / sp[[-3.0, 3.0]].mean(axis=1)
print("Speed at lag 0 over speed at +-3 s, median", round(float(ratio.median()), 2), "| range", round(float(ratio.min()), 2), "to", round(float(ratio.max()), 2))

lfp_lags = list(lshpc.LFP_LAGS_MS)
lk = pd.DataFrame([r.get("clock_lfp", {}) for r in results], index=S.name)
ok = lk.notna().all(axis=1)
lk_peak = lk[ok].idxmax(axis=1)
print("\nTheta phase locking (mean resultant length) of hippocampal spikes, LFP shifted by each lag (ms):")
print(pd.concat([lk[ok].round(3), lk_peak.rename("peak")], axis=1).to_string())
print("Files per peak lag (ms):", lk_peak.value_counts().sort_index().to_dict())
print("Locking at 0 ms over locking at +-200 ms, median",
      round(float((lk[ok][0] / lk[ok][[-200, 200]].mean(axis=1)).median()), 2))

rt = pd.DataFrame([r["summary"]["clock_trial_rate"] or {} for r in results], index=S.name).dropna()
rt_peak = rt.idxmax(axis=1)
print("\nMean firing rate of all units in the trial windows shifted by each lag, files per peak lag:",
      rt_peak.value_counts().sort_index().to_dict())

Three clock checks, each a contrast between the true and shifted alignments.

- Positions against the trials table: running speed inside the trial windows peaks at a lag of 0
  or 0.5 s in 65 of 67 files and is a median 2.2 times its value at 3 s, so positions and trials share a
  clock to within about half a second. In DT9 20170501 speed does not change with the lag (31.5
  cm/s at 0 against 29.1 to 32.8 at other lags): its trials do not match its positions, and it is
  left out by the condition rule (no condition reaches 10 trials). In one DT2 file (20160221) speed
  peaks at a lag of 1 s; its trials may be offset from its positions by up to a second. This is
  consistent with the data, not proven for every trial.
- Spikes against the LFP: theta phase locking of hippocampal spikes peaks within 75 ms of zero lag
  in 44 of the 47 files with hippocampal units and a theta channel, and at 200 ms it is lower by a
  median of about a sixth (ratio 1.2). The contrast is weak: theta has a period of about 125 ms, so
  a shift of 200 ms is under two cycles. This is consistent with a shared clock to within about 100
  ms, and does not prove it. It could not be checked in the 17 DT8 files and the three DT2 and
  DT7 files without labelled hippocampal units.
- Spikes against the trials: the mean rate of all units in the trial windows changes little with
  the lag (running modulates many LS units weakly or negatively), so this contrast does not settle
  the alignment of spikes and trials. It is not established for the LS-only sessions beyond the
  spike-LFP check of the other files.

In [ ]:
print("processing/behavior/allocentric_frame_tracking, per subject:")
for subj, g in S.groupby("subject"):
    held = [d for d in g.allocentric_frame_tracking if d]
    print(f"  {subj}: in {len(held)} of {len(g)} files;", held[0] if held else "")

`processing/behavior/allocentric_frame_tracking` holds one series, `orientation`: a head-orientation
quaternion (x, y, z, w) with its timestamps, in every file of DT5, DT7, DT8 and DT9 and in none of
DT2. Its stated unit is metres, which does not apply to a quaternion. Nothing in it, or in the
trials table, marks a maze rotation, so figs6-reference-frame stays blocked.

## LS neurons recorded (ls-neuron-count)

> A total of 1,647 LS neurons were recorded from five animals, in 62 sessions.

In [ ]:
ls_units = units["ls"]
print("LS units per subject:", ls_units.drop("all").to_dict(), "| total", int(ls_units["all"]))
n_ls_sessions = sum("ls" in d for d in S.unit_locations)
print("Sessions with LS units:", n_ls_sessions, "| subjects with LS units:", int((ls_units.drop("all") > 0).sum()))
print("Paper: 1,647 LS neurons, 62 sessions, five animals. Difference in units:",
      int(ls_units["all"]) - 1647, f"({(int(ls_units['all']) - 1647) / 1647:+.1%})")
print("Unlabelled units (DT7 20170317), not counted:", int(units.loc["all"].get("unlabelled", 0)))

**Result: reproduced.** 1,696 units are labelled LS, 3.0% more than the paper's 1,647 and within tolerance, in 62 sessions as in the paper, and all five rats hold LS units.

## Hippocampal units (hpc-unit-count)

> The hippocampal recordings hold 2,297 isolated CA1 units and 654 CA3 units (2,951 in all).

In [ ]:
ca1, ca3 = int(units.loc["all", "hpc"]), int(units.loc["all", "ca3"])
print("Units labelled hpc (taken as CA1) per subject:", units["hpc"].drop("all").to_dict(), "| total", ca1)
print("Units labelled ca3 per subject:", units["ca3"].drop("all").to_dict(), "| total", ca3)
print(f"CA1: {ca1} against 2,297 ({(ca1 - 2297) / 2297:+.1%}); CA3: {ca3} against 654 ({(ca3 - 654) / 654:+.1%}); "
      f"sum {ca1 + ca3} against 2,951 ({(ca1 + ca3 - 2951) / 2951:+.1%})")
print("Sessions with hpc units:", int(S.has_hpc_units.sum()) - int(S.has_ca3_units.sum()), "| with ca3 units:", int(S.has_ca3_units.sum()),
      "| paper, Figure 6A: 63 CA1 and 19 CA3 sessions; 45 hippocampal sessions in Figure 1I")

**Result: partially reproduced.** 2,593 units are labelled hpc against 2,297 CA1 units (+12.9%, outside tolerance) and 624 are labelled ca3 against 654 (-4.6%, within). Their sum, 3,217, is 9.0% above 2,951. The labels hpc and ca3 are read as CA1 and CA3; the dandiset does not say that hpc means CA1.

## LS neurons passing the activity criterion (ls-active-neurons)

> Of the LS neurons, 840 fired at least 1.5 action potentials per trial on average.

In [ ]:
ls_all = C[C.region == "ls"]
def active_neurons(df, k):
    return df[df.n_spikes >= k * df.n_trials].groupby("neuron").ngroups
n_active = LS.neuron.nunique()
print("LS units in a condition with at least 10 trials:", ls_all.neuron.nunique(), "of", int(units.loc["all", "ls"]))
print("Active LS neurons (1.5 spikes per trial in at least one condition):", n_active, "| paper 840")
print("Per subject:", LS.groupby("subject").neuron.nunique().to_dict())
print("Included LS conditions (neuron and condition pairs):", len(LS), "| paper 4,393 conditions")
for k in (1.0, 1.5, 2.0, 3.0):
    print(f"  threshold {k} spikes per trial: {active_neurons(ls_all, k)} neurons")

**Result: reproduced.** 793 LS neurons fire at least 1.5 spikes per trial in a condition with at least 10 trials, against 840 (-5.6%, within tolerance), over 3,694 neuron-condition pairs (the paper has 4,393). The count depends on how conditions are formed, which the dandiset does not record.

## Phase coding against shuffled data (fig2-ls-phase-vs-shuffle)

> The majority of LS neurons (751 of 840, 89%) showed position encoding in their spike theta phase,
> compared with shuffled data, in at least one condition.

In [ ]:
def frac_neurons(df, mask):
    return mask.groupby(df.neuron).any()

sig_chance = (LSd.p_phase_vs_chance < 0.01) & (LSd.mse_phase < LSd.mse_chance_phase)
per_neuron = frac_neurons(LSd, sig_chance)
print(f"Active LS neurons decoded: {LSd.neuron.nunique()} (dropped {n_active - LSd.neuron.nunique()} in the file without theta reference)")
print(f"Neurons with phase MSE below chance at p < 0.01 in at least one condition: {int(per_neuron.sum())} "
      f"of {len(per_neuron)} = {per_neuron.mean():.3f} | paper 751 of 840 = 0.894")
print("Conditions significant:", f"{sig_chance.mean():.3f}", "of", len(LSd))
print("Per subject:", per_neuron.groupby([S.subject[n[0]] for n in per_neuron.index]).mean().round(3).to_dict())
for p in (0.05, 0.01, 0.001):
    m = (LSd.p_phase_vs_chance < p) & (LSd.mse_phase < LSd.mse_chance_phase)
    print(f"  p < {p}: {frac_neurons(LSd, m).mean():.3f} of neurons")
print("Mean MSE (bins^2) over LS conditions: phase", round(LSd.mse_phase.mean()), "| chance phase", round(LSd.mse_chance_phase.mean()),
      "| rate", round(LSd.mse_rate.mean()), "| chance rate", round(LSd.mse_chance_rate.mean()),
      "| uniform guessing over 200 bins", round(2 * (200**2 - 1) / 12))
rate_sig = (LSd.p_rate_vs_chance < 0.01) & (LSd.mse_rate < LSd.mse_chance_rate)
print(f"For comparison, rate MSE below chance at p < 0.01: {frac_neurons(LSd, rate_sig).mean():.3f} of neurons")
good_theta = S.session[S.theta_ratio > 1]
m1 = LSd.session.isin(good_theta)
pn = frac_neurons(LSd[m1], sig_chance[m1])
print(f"Variant, sessions whose theta channel has more 6-9 Hz than 1-4 Hz power ({LSd[m1].session.nunique()} sessions): "
      f"{pn.mean():.3f} of {len(pn)} neurons; other sessions: {frac_neurons(LSd[~m1], sig_chance[~m1]).mean():.3f}")

**Result: not reproduced.** 52.1% of the decoded active LS neurons (408 of 783) decode position from phase better than chance at p < 0.01 in at least one condition, against 89% in the paper. With p < 0.05 the fraction is 66.0%, still outside tolerance; at p < 0.01 the rate code passes the same test against its own chance in 54.4% of neurons. In the 37 sessions whose theta channel has more 6 to 9 Hz than 1 to 4 Hz power the fraction is 52.9%, so a weak theta reference does not account for the gap. The decoder and the phase maps are rebuilt from the Methods and code, without the paper's linearization, conditions or theta channel.

## Phase decoding better than rate decoding (fig2-ls-phase-beats-rate)

> Spike-phase decoding outperformed rate decoding of the rat's position for 786 of 840 LS neurons,
> 3,046 of 4,393 conditions, and 62 of 62 sessions.

In [ ]:
better = (LSd.p_phase_vs_rate < 0.01) & (LSd.mse_phase < LSd.mse_rate)
nb = frac_neurons(LSd, better)
ses = LSd.groupby("session")[["mse_phase", "mse_rate"]].mean()
print(f"Neurons: {int(nb.sum())} of {len(nb)} = {nb.mean():.3f} | paper 786 of 840 = 0.936")
print(f"Conditions: {int(better.sum())} of {len(better)} = {better.mean():.3f} | paper 3,046 of 4,393 = 0.693")
print(f"Sessions with lower mean phase than rate MSE: {int((ses.mse_phase < ses.mse_rate).sum())} of {len(ses)} = "
      f"{(ses.mse_phase < ses.mse_rate).mean():.3f} | paper 62 of 62")
worse = (LSd.p_phase_vs_rate < 0.01) & (LSd.mse_phase > LSd.mse_rate)
print(f"Conditions where rate is better at p < 0.01: {worse.mean():.3f}")
by = pd.DataFrame({"neurons": nb.groupby([S.subject[n[0]] for n in nb.index]).mean(),
                   "conditions": better.groupby(LSd.subject).mean(),
                   "sessions": (ses.mse_phase < ses.mse_rate).groupby(S.subject[ses.index].values).mean()})
print("\nPer subject (fractions):")
print(by.round(3).to_string())
for p in (0.05, 0.001):
    m = (LSd.p_phase_vs_rate < p) & (LSd.mse_phase < LSd.mse_rate)
    print(f"  p < {p}: neurons {frac_neurons(LSd, m).mean():.3f}, conditions {m.mean():.3f}")
bt = (LSd.p_phase_vs_rate_trialsplit < 0.01) & (LSd.mse_phase_trialsplit < LSd.mse_rate_trialsplit)
print(f"Variant, whole trials held out: neurons {frac_neurons(LSd, bt).mean():.3f}, conditions {bt.mean():.3f}")
nb1 = frac_neurons(LSd[m1], better[m1])
print(f"Variant, sessions with theta ratio above 1: neurons {nb1.mean():.3f} of {len(nb1)}, conditions {better[m1].mean():.3f} of {int(m1.sum())}")

**Result: not reproduced.** Phase beats rate at p < 0.01 in 84.5% of neurons (paper 93.6%) and 48.1% of conditions (paper 69.3%), both outside tolerance, and in 58 of 58 sessions (paper 62 of 62). As fewer than half the conditions pass, the criterion's partial outcome does not apply. The direction holds in every rat and session. In the 37 sessions with a theta ratio above 1, 87.5% of neurons and 52.8% of conditions pass. Samples, not whole trials, are split at random, as the paper's per-cell decoder does (`randperm` over all trial-by-bin samples in scratch_07_11_17_spatialBinnedMaxCorr.m); with whole trials held out the fractions fall to 72.9% and 35.4%.

## Phase MSE lower than rate MSE (fig2e-ls-phase-lower-mse)

> Across LS neurons the spike-phase code is stronger than the rate code (two-sample t tests;
> N = 4,393 conditions, p < 1e-100; N = 62 sessions, p < 1e-10).

In [ ]:
t_c = stats.ttest_ind(LSd.mse_phase, LSd.mse_rate)
t_s = stats.ttest_ind(ses.mse_phase, ses.mse_rate)
print(f"Conditions (N = {len(LSd)}): phase {LSd.mse_phase.mean():.0f}, rate {LSd.mse_rate.mean():.0f}; t = {t_c.statistic:.1f}, p = {t_c.pvalue:.2e}")
print(f"Sessions (N = {len(ses)}): phase {ses.mse_phase.mean():.0f}, rate {ses.mse_rate.mean():.0f}; t = {t_s.statistic:.1f}, p = {t_s.pvalue:.2e}")
print("Per subject, conditions:")
for subj, g in LSd.groupby("subject"):
    t = stats.ttest_ind(g.mse_phase, g.mse_rate)
    print(f"  {subj}: N = {len(g)}, phase {g.mse_phase.mean():.0f}, rate {g.mse_rate.mean():.0f}, p = {t.pvalue:.1e}")
tg = stats.ttest_ind(LSd[m1].mse_phase, LSd[m1].mse_rate)
print(f"Variant, sessions with theta ratio above 1: phase {LSd[m1].mse_phase.mean():.0f}, rate {LSd[m1].mse_rate.mean():.0f}, p = {tg.pvalue:.1e}")
tt = stats.ttest_ind(LSd.mse_phase_trialsplit, LSd.mse_rate_trialsplit)
print(f"Variant, whole trials held out: phase {LSd.mse_phase_trialsplit.mean():.0f}, rate {LSd.mse_rate_trialsplit.mean():.0f}, p = {tt.pvalue:.1e}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
bins = np.arange(2000, 14001, 250)
ax[0].hist(LSd.mse_phase, bins, alpha=.6, label="phase"); ax[0].hist(LSd.mse_rate, bins, alpha=.6, label="rate")
ax[0].set_xlabel("MSE (bins$^2$)"); ax[0].set_ylabel("LS conditions"); ax[0].legend()
d = LSd.mse_phase - LSd.mse_rate
ax[1].hist(d, np.arange(-6000, 6001, 200), color="m"); ax[1].set_xlabel("MSE phase - rate (bins$^2$)")
plt.tight_layout(); plt.show()
print("Outside the axes: MSE", int(((LSd.mse_phase < 2000) | (LSd.mse_phase > 14000)).sum()), "phase and",
      int(((LSd.mse_rate < 2000) | (LSd.mse_rate > 14000)).sum()), "rate values; difference", int((d.abs() > 6000).sum()),
      "| ranges: phase", LSd.mse_phase.min().round(), "to", LSd.mse_phase.max().round(), ", rate", LSd.mse_rate.min().round(), "to", LSd.mse_rate.max().round())

**Result: reproduced.** Over 3,660 LS conditions the phase MSE (6,359) is below the rate MSE (7,404), t = -33.8, p = 2.5e-232, below the paper's 1e-100; over 58 sessions p = 2.9e-23, below 1e-10. It holds in each rat (DT7 only at p = 0.018), in the sessions with a theta ratio above 1 (p = 2.9e-147) and with whole trials held out.

## Hippocampal place fields, rate and phase decoding (fig1i-hpc-rate-phase-equivalent)

> For hippocampal place fields the distributions of rate and phase decoding errors are
> statistically equivalent (two-sample KS test; 1,820 place fields, p = 0.19; 45 sessions, p = 0.61).

In [ ]:
PF = HPd[HPd.n_fields > 0]
hs = PF.groupby("session")[["mse_phase", "mse_rate"]].mean()
k1 = stats.ks_2samp(PF.mse_phase, PF.mse_rate)
k2 = stats.ks_2samp(hs.mse_phase, hs.mse_rate)
print(f"Place fields: N = {len(PF)} (paper 1,820); phase {PF.mse_phase.mean():.0f}, rate {PF.mse_rate.mean():.0f}; "
      f"KS D = {k1.statistic:.3f}, p = {k1.pvalue:.3f}; mean difference {(PF.mse_phase - PF.mse_rate).mean():.0f}")
print(f"Sessions: N = {len(hs)} (paper 45); KS D = {k2.statistic:.3f}, p = {k2.pvalue:.3f}; mean difference {(hs.mse_phase - hs.mse_rate).mean():.0f}")
print("Per subject (fields):")
for subj, g in PF.groupby("subject"):
    k = stats.ks_2samp(g.mse_phase, g.mse_rate)
    print(f"  {subj}: N = {len(g)}, phase - rate {(g.mse_phase - g.mse_rate).mean():.0f}, KS p = {k.pvalue:.3f}")
k3 = stats.ks_2samp(HPd.mse_phase, HPd.mse_rate)
print(f"Variant, all included hippocampal conditions (N = {len(HPd)}): phase - rate {(HPd.mse_phase - HPd.mse_rate).mean():.0f}, KS p = {k3.pvalue:.2e}")
k4 = stats.ks_2samp(PF.mse_phase_trialsplit, PF.mse_rate_trialsplit)
print(f"Variant, whole trials held out: phase - rate {(PF.mse_phase_trialsplit - PF.mse_rate_trialsplit).mean():.0f}, KS p = {k4.pvalue:.3f}")
PFb = HPd[HPd.n_fields_box5 > 0]
k5 = stats.ks_2samp(PFb.mse_phase, PFb.mse_rate)
print(f"Variant, fields of the 5-bin box map: N = {len(PFb)}, phase - rate {(PFb.mse_phase - PFb.mse_rate).mean():.0f}, KS p = {k5.pvalue:.3f}")
PFn = HPd[HPd.n_fields_nosd > 0]
k6 = stats.ks_2samp(PFn.mse_phase, PFn.mse_rate)
print(f"Variant, fields without the rule dropping peaks whose SD across trials exceeds their mean: N = {len(PFn)}, "
      f"phase - rate {(PFn.mse_phase - PFn.mse_rate).mean():.0f}, KS D = {k6.statistic:.3f}, p = {k6.pvalue:.3f}")

**Result: not reproduced.** Over 531 hippocampal place fields the KS test finds a difference between phase and rate MSE (D = 0.085, p = 0.044), where the paper finds none (p = 0.19); over 42 sessions it finds none (p = 0.61). The phase MSE is lower by 234 bins squared on average over fields. The result sits near the threshold and depends on the set of fields: with the 318 fields of the 5-bin box map p = 0.097, and per rat only DT2 differs (p = 0.001). Without the code's rule dropping peaks whose SD across trials exceeds their mean, 2,320 fields are found, more than the paper's 1,820, and the distributions differ more clearly (D = 0.104, p below 0.001). Only 531 fields are found here, against 1,820 (see ls-few-place-fields); over all hippocampal conditions the distributions differ (p = 7.4e-48).

## Mean phase minus rate MSE (fig2fg-mse-difference)

> The mean phase minus rate MSE is about -1,000 for LS conditions and -33 for hippocampal place
> fields, and -1,024 and -25 for LS and hippocampal sessions.

In [ ]:
vals = {"LS conditions": (LSd.mse_phase - LSd.mse_rate).mean(), "hippocampal place fields": (PF.mse_phase - PF.mse_rate).mean(),
        "LS sessions": (ses.mse_phase - ses.mse_rate).mean(), "hippocampal sessions": (hs.mse_phase - hs.mse_rate).mean()}
paper = {"LS conditions": -1000, "hippocampal place fields": -33, "LS sessions": -1024, "hippocampal sessions": -25}
for k, v in vals.items():
    print(f"{k}: {v:.0f} (paper {paper[k]}, tolerance +-{abs(paper[k]) * 0.1:.1f})")
print("\nPer subject, LS conditions and hippocampal place fields:")
print(pd.DataFrame({"LS": (LSd.mse_phase - LSd.mse_rate).groupby(LSd.subject).mean(),
                    "HPC fields": (PF.mse_phase - PF.mse_rate).groupby(PF.subject).mean()}).round(0).to_string())
print("\nVariant, whole trials held out: LS", round((LSd.mse_phase_trialsplit - LSd.mse_rate_trialsplit).mean()),
      "| hippocampal fields", round((PF.mse_phase_trialsplit - PF.mse_rate_trialsplit).mean()))

**Result: partially reproduced.** The LS condition mean, -1,045, is within tolerance of -1,000; the LS session mean, -1,132, is just outside -1,024 +- 102; the hippocampal values, -234 for place fields and -126 for sessions, are far from -33 and -25. LS values are negative and larger in magnitude than the hippocampal ones at both levels.

## Place fields in LS and hippocampus (ls-few-place-fields)

> Only 5.3% (232 of 4,393) of LS rate maps showed place fields by the criteria applied to
> hippocampal units, where 31.7% (1,820 of 5,735) did.

In [ ]:
f_ls, f_hp = (LS.n_fields > 0), (HP.n_fields > 0)
print(f"LS rate maps with a field: {int(f_ls.sum())} of {len(LS)} = {f_ls.mean():.3f} (paper 0.053)")
print(f"Hippocampal rate maps with a field: {int(f_hp.sum())} of {len(HP)} = {f_hp.mean():.3f} (paper 0.317)")
tab = [[f_ls.sum(), (~f_ls).sum()], [f_hp.sum(), (~f_hp).sum()]]
chi = stats.chi2_contingency(tab)
print(f"Chi-squared {chi.statistic:.1f}, p = {chi.pvalue:.2e}")
print(pd.DataFrame({"LS": f_ls.groupby(LS.subject).mean(), "HPC": f_hp.groupby(HP.subject).mean(),
                    "n LS": f_ls.groupby(LS.subject).size(), "n HPC": f_hp.groupby(HP.subject).size()}).round(3).to_string())
print("\nVariants of the rate map and threshold:")
for col, name in (("n_fields", "Gaussian SD 4 bins, 20% threshold (main)"), ("n_fields_gauss5", "Gaussian SD 5 bins, 20%"),
                  ("n_fields_box5", "5-bin box map, 20%"), ("n_fields_thresh01", "Gaussian SD 4 bins, 1% (the code's default)"),
                  ("n_fields_nosd", "Gaussian SD 4 bins, 20%, without the mean < SD exclusion")):
    a, b = LS[col] > 0, HP[col] > 0
    c2 = stats.chi2_contingency([[a.sum(), (~a).sum()], [b.sum(), (~b).sum()]])
    print(f"  {name}: LS {a.mean():.3f}, hippocampus {b.mean():.3f}, chi-squared p = {c2.pvalue:.1e}")

**Result: partially reproduced.** 5.4% of LS rate maps have a place field, within tolerance of 5.3%, and fewer than in the hippocampus (chi-squared p = 8.9e-18), but only 10.5% of hippocampal rate maps have a field, against 31.7%. The hippocampal fraction is 8.0% to 13.6% per rat. With a Gaussian of SD 5 bins the fractions are 4.9% and 11.6%; with the 5-bin box map 5.8% and 6.3%, which do not differ (p = 0.40); with the 1% threshold of the code's default, 0.4% and 4.6%. The rule of bz_getPlaceFields1D that drops peaks whose SD across trials exceeds their mean accounts for most of the hippocampal shortfall: without it the fractions are 20.0% (LS) and 46.0% (hippocampus), so LS is then far above 5.3% and the hippocampus above 31.7%. Neither setting matches both of the paper's fractions. The rule is strict here probably because the rebuilt linearization gives about one position sample per bin and trial, which makes per-trial rates noisy.

## Properties of hippocampal place fields (hpc-place-field-properties)

> 711 of 1,425 hippocampal neurons had at least one place field (1.27 fields per neuron on
> average); the median peak in-field rate was 8.5 Hz and the median field width 38 bins.

In [ ]:
has = (HP.n_fields > 0).groupby(HP.neuron).any()
peaks = np.concatenate(HP.field_peaks_hz.dropna().values)
widths = np.concatenate(HP.field_widths.dropna().values)
nf = HP.groupby("neuron").n_fields.sum()
print(f"Hippocampal neurons with a field: {int(has.sum())} of {len(has)} = {has.mean():.3f} (paper 711 of 1,425 = 0.499)")
print(f"Fields per active neuron: mean {nf.mean():.2f} (paper 1.27)")
print(f"Median peak rate {np.median(peaks):.1f} Hz (paper 8.5); median width {np.median(widths):.0f} bins (paper 38); N = {len(peaks)} fields")
print("Per subject: fraction with a field", has.groupby([S.subject[n[0]] for n in has.index]).mean().round(3).to_dict())
for col, name in (("n_fields_gauss5", "Gaussian SD 5"), ("n_fields_box5", "5-bin box map"), ("n_fields_thresh01", "1% threshold"), ("n_fields_nosd", "no mean < SD exclusion")):
    print(f"Variant, {name}: {(HP[col] > 0).groupby(HP.neuron).any().mean():.3f} of neurons with a field")

**Result: partially reproduced.** The median peak rate, 8.4 Hz, and the median field width, 39 bins, are within tolerance of 8.5 Hz and 38 bins; but 22.9% of active hippocampal neurons have a field, against 49.9%, with 0.47 fields per neuron against 1.27. Under the variants of the map and threshold the fraction is 12.4% to 24.1%, and 62.3% without the rule dropping peaks whose SD across trials exceeds their mean (see ls-few-place-fields).

## Stability of LS and hippocampal place fields (ls-place-fields-unstable)

> The few LS rate maps with a place field were less stable across trials than CA1/CA3 place fields
> (median coefficient of variance 41.5 against 10; two-sample t test, p < 1e-20).

In [ ]:
lf_, hf_ = LS[LS.n_fields > 0], HP[HP.n_fields > 0]
for name, col, scale in (("(a) SD over trials at the field peak, Hz", "field_peak_sd", 1), ("(b) SD over mean at the peak, %", "field_peak_cv", 100)):
    a, b = lf_[col] * scale, hf_[col] * scale
    t = stats.ttest_ind(a, b)
    print(f"{name}: LS median {a.median():.1f} (N = {len(a)}), hippocampus median {b.median():.1f} (N = {len(b)}); t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
print("Paper: 41.5 against 10.")

**Result: not reproduced.** Under neither reading are LS fields more variable than hippocampal ones. Under reading (a), the SD at the peak that makeFig2.m computes, the LS median (3.5 Hz) is below the hippocampal one (6.4 Hz, t = -7.76, p = 2.8e-14), the reverse of the claim. Under reading (b), SD over mean, the medians are 86.2% and 83.6% and do not differ (p = 0.12). Both readings are far from 41.5 and 10.

## Smoothing scale of peak information (fig1e-phase-info-smaller-scale)

> Hippocampal place fields reach their peak information at smaller smoothing scales for phase than
> for rate (medians 15 and 29 bins; two-sample t test, p < 1e-40).

In [ ]:
info = [dict(x, session=i, subject=results[i]["summary"]["subject"]) for i, r in enumerate(results) for x in r.get("info", [])]
def smooth5(v):
    return lshpc.matlab_smooth(v, 5)
best_p = np.array([np.argmax(smooth5(x["phase"])) + 1 for x in info])
best_r = np.array([np.argmax(smooth5(x["rate"])) + 1 for x in info])
keep_p, keep_r = best_p < 49, best_r < 49
t = stats.ttest_ind(best_p[keep_p], best_r[keep_r])
print(f"Place fields with information curves: {len(info)}; kept (best smoothing below 49): phase {keep_p.sum()}, rate {keep_r.sum()}")
print(f"Median best smoothing: phase {np.median(best_p[keep_p]):.0f} bins, rate {np.median(best_r[keep_r]):.0f} bins (paper 15 and 29); "
      f"means {best_p[keep_p].mean():.1f} and {best_r[keep_r].mean():.1f}; t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
print(f"Fields left out at 49 or more: phase {(~keep_p).sum()}, rate {(~keep_r).sum()}")
subj = np.array([x["subject"] for x in info])
for s_ in sorted(set(subj)):
    m = subj == s_
    print(f"  {s_}: N = {m.sum()}, median phase {np.median(best_p[m & keep_p]) if (m & keep_p).any() else np.nan}, "
          f"rate {np.median(best_r[m & keep_r]) if (m & keep_r).any() else np.nan}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(best_p[keep_p], np.arange(0, 51, 2), alpha=.6, label="phase"); ax.hist(best_r[keep_r], np.arange(0, 51, 2), alpha=.6, label="rate")
ax.set_xlabel("smoothing with peak information (bins)"); ax.set_ylabel("place fields"); ax.legend(); plt.show()
print("Plotted values: phase", best_p[keep_p].min(), "to", best_p[keep_p].max(), "| rate", best_r[keep_r].min(), "to", best_r[keep_r].max(), "bins; none outside the axes")

**Result: partially reproduced.** Phase information peaks at smaller smoothing than rate information (medians 10 and 17 bins, t = -11.18, p = 1.9e-27), and in each rat, but both medians are below the paper's 15 and 29, and p does not reach 1e-40 with 531 fields. At the same nominal span the rate and phase smoothings differ slightly (MATLAB smooth drops an even span by one; circ_smoothTS uses 2 ceil(w/2) + 1 bins), as in the paper's code.

## Information of phase and rate at fine and coarse scales (fig1g-phase-more-info-fine-scale)

> At smoothing below 6 cm (about one theta cycle) phase carries more spatial information than rate
> in hippocampal place fields, and above 10 cm rate carries more.

In [ ]:
P = np.array([x["phase"] / x["n_trials"] for x in info])
R = np.array([x["rate"] / x["n_trials"] for x in info])
rows = []
for w in lshpc.INFO_SMOOTHINGS:
    t = stats.ttest_ind(P[:, w - 1], R[:, w - 1])
    rows.append({"smoothing": w, "phase_minus_rate": (P[:, w - 1] - R[:, w - 1]).mean(), "p": t.pvalue})
D = pd.DataFrame(rows)
print(D[D.smoothing.isin([1, 2, 3, 4, 5, 6, 7, 10, 15, 20, 30, 40, 50])].to_string(index=False))
fine = D[D.smoothing.isin([2, 3])]
coarse = D[D.smoothing >= 7]
print(f"\nFine (2 and 3 bins): phase above rate at p < 1e-5 in {int(((fine.phase_minus_rate > 0) & (fine.p < 1e-5)).sum())} of 2")
print(f"Coarse (7 to 50 bins): rate above phase at p < 0.05 in {int(((coarse.phase_minus_rate < 0) & (coarse.p < 0.05)).sum())} of {len(coarse)} "
      f"({((coarse.phase_minus_rate < 0) & (coarse.p < 0.05)).mean():.2f})")
print("Bin size: the median trial path over 200 bins is", round(float(np.nanmedian(S[S.position_unit_used == "cm"].path_length_median)) / 200, 2),
      "cm in the files with a scale (paper about 1.5 cm)")

**Result: partially reproduced.** At coarse smoothing (7 to 50 bins) rate carries more information than phase at p < 0.05 in 42 of 44 smoothings (95%, above the criterion's 80%), as the paper reports above 10 cm. At fine smoothing phase carries more, but not at p < 1e-5 (2 bins, +0.018, p = 2.1e-5; 3 bins, +0.006, p = 0.16). Bins here are about 1.1 cm, so 2 and 3 bins are below 6 cm.

## Phase coding and LS depth (fig4d-depth-gradient)

> The strength of phase coding relative to rate coding increases with recording depth within the LS,
> within sessions and across animals.

In [ ]:
sd = LSd.groupby("session").apply(lambda g: (g.mse_phase - g.mse_rate).mean()).rename("diff").to_frame()
sd["subject"] = S.subject[sd.index].values
sd["depth"] = S.ls_depth[sd.index].values
rows = []
for subj, g in sd.groupby("subject"):
    if len(g) >= 4 and g.depth.nunique() > 1:
        r, p = stats.pearsonr(g.depth, g["diff"])
    else:
        r, p = np.nan, np.nan
    rows.append({"subject": subj, "sessions": len(g), "depth_range": f"{g.depth.min()}-{g.depth.max()}", "r": r, "p": p})
print(pd.DataFrame(rows).round(3).to_string(index=False))
c = sd.copy()
c["depth"] -= c.groupby("subject").depth.transform("mean")
c["diff"] -= c.groupby("subject")["diff"].transform("mean")
r, p = stats.pearsonr(c.depth, c["diff"])
print(f"Pooled, centred within animal: r = {r:.3f}, p = {p:.3f}, N = {len(c)} sessions")
r2, p2 = stats.pearsonr(sd.depth, sd["diff"])
print(f"Variant, pooled without centring: r = {r2:.3f}, p = {p2:.3f}")
sd2 = sd.copy(); sd2["depth"] = S.hpc_depth[sd.index].values
for subj, g in sd2.groupby("subject"):
    if len(g) >= 4 and g.depth.nunique() > 1:
        print(f"  variant with the second depth, {subj}: r = {stats.pearsonr(g.depth, g['diff'])[0]:.3f}")
c2 = sd2.copy()
c2["depth"] -= c2.groupby("subject").depth.transform("mean")
c2["diff"] -= c2.groupby("subject")["diff"].transform("mean")
r3, p3 = stats.pearsonr(c2.depth, c2["diff"])
print(f"Variant, second depth, pooled and centred within animal: r = {r3:.3f}, p = {p3:.3f}")

**Result: partially reproduced.** With depth and value centred within each rat, deeper LS sessions have more negative phase minus rate MSE (r = -0.365, p = 0.005, 58 sessions). Per rat r is negative in DT2 (-0.70), DT5 (-0.43) and DT9 (-0.62) but positive in DT8 (+0.24), so it does not hold in every rat. With the second depth of the name the per-rat signs are the same and the pooled r is -0.268 (p = 0.042); which depth is the LS probe's is an assumption (see the data check). The within-session part cannot be tested, as the dandiset gives no recording site per unit.

## Onset phase of LS phase fields and depth (fig4e-onset-depth)

> The theta phase at the onset of LS phase fields varies with recording depth (DT5 r = 0.38, DT9 r = 0.25,
> DT2 r = 0.27; circular-linear correlation, p < 0.001).

In [ ]:
sig = LSd[sig_chance & LSd.onset_phase.notna()]
best = sig.loc[sig.groupby("neuron").mse_phase.idxmin()]
best = best.assign(depth=S.ls_depth[best.session].values)
print(f"Neurons with significant phase coding and an onset phase: {len(best)}")
for subj in ["DT5", "DT9", "DT2", "DT7", "DT8"]:
    g = best[best.subject == subj]
    if len(g) > 5 and g.depth.nunique() > 1:
        r, p = lshpc.circ_corrcl(g.onset_phase.values, g.depth.values)
        print(f"  {subj}: N = {len(g)} neurons, {g.session.nunique()} sessions, depths {g.depth.min()}-{g.depth.max()} um, r = {r:.3f}, p = {p:.2e}")
print("Paper: DT5 0.38, DT9 0.25, DT2 0.27, each p < 0.001.")
allc = sig.assign(depth=S.ls_depth[sig.session].values)
print("Variant, every significant condition:")
for subj in ["DT5", "DT9", "DT2"]:
    g = allc[allc.subject == subj]
    r, p = lshpc.circ_corrcl(g.onset_phase.values, g.depth.values)
    print(f"  {subj}: N = {len(g)}, r = {r:.3f}, p = {p:.2e}")
print("Variant, sessions whose theta channel has the rat's most common label (one reference region per rat):")
lab = S.theta_label[best.session].values
for subj in ["DT5", "DT9", "DT2"]:
    g = best[(best.subject == subj)]
    gl = lab[best.subject == subj]
    top = pd.Series(gl).value_counts(dropna=False).index[0]
    g = g[pd.Series(gl).eq(top).values] if top == top else g
    r, p = lshpc.circ_corrcl(g.onset_phase.values, g.depth.values)
    print(f"  {subj}: label {top}, N = {len(g)} neurons, {g.session.nunique()} sessions, depths {g.depth.min()}-{g.depth.max()} um, "
          f"r = {r:.3f}, p = {p:.2e}")

**Result: not reproduced.** The circular-linear correlation between onset phase and LS depth is r = 0.082 (p = 0.57) in DT5, 0.072 (p = 0.78) in DT9 and 0.118 (p = 0.65) in DT2, against 0.38, 0.25 and 0.27 at p < 0.001. Using every significant condition gives the same, and so does keeping only the sessions whose theta channel has the rat's most common label (hpc in DT5 and DT9; r = 0.027 and 0.170, p = 0.96 and 0.37), which removes the switches between hpc and ca3 references. The test rests on the session depth (the first number in the session name) and on a theta channel chosen per session, which may differ in layer between sessions. That the first depth is the LS depth is an assumption (see fig4d-depth-gradient).

## LS-hippocampal theta coherence along the run (fig6c-ca3-coherence-increases)

> Theta phase coherence between LS and the CA3 pyramidal layer increases from the beginning to the
> end of the run, while coherence between LS and CA1 stays relatively flat.

In [ ]:
rows = []
for i, r in enumerate(results):
    f6 = r.get("fig6")
    if not f6:
        continue
    sm = r["summary"]
    labels = sm["unit_locations"]
    groups = (["CA3"] if "ca3" in labels else []) + (["CA1"] if ("hpc" in labels or "ca1" in labels) else [])
    groups = groups or ["none"]   # no hippocampal units: left out of the main test, kept for a variant
    seg = np.arange(lshpc.N_SEGMENTS)
    ok = np.isfinite(f6["coherence"])
    ok2 = np.isfinite(f6["cfc"])
    for group in groups:
        rows.append({"session": i, "subject": sm["subject"], "group": group, "channel": S.theta_label[i],
                     "coh_mean": np.nanmean(f6["coherence"]), "coh_slope": np.polyfit(seg[ok], f6["coherence"][ok], 1)[0] if ok.sum() > 2 else np.nan,
                     "cfc_mean": np.nanmean(f6["cfc"]), "cfc_slope": np.polyfit(seg[ok2], f6["cfc"][ok2], 1)[0] if ok2.sum() > 2 else np.nan,
                     "coh": f6["coherence"], "cfc": f6["cfc"]})
F6all = pd.DataFrame(rows)
F6 = F6all[F6all.group != "none"]
print("Sessions by group and subject (CA3: units labelled ca3; CA1: units labelled hpc):")
print(F6.groupby(["group", "subject"]).size().to_string())
print("Left out, no hippocampal units:", F6all[F6all.group == "none"].groupby("subject").size().to_dict())
print("Label of the hippocampal channel by group:", F6.groupby("group").channel.value_counts().to_dict())
for g in ("CA3", "CA1"):
    x = F6[F6.group == g].coh_slope.dropna()
    w = stats.wilcoxon(x)
    print(f"{g}: N = {len(x)} sessions, median coherence slope {x.median():.4f} per segment, signed-rank p = {w.pvalue:.3f}, positive in {(x > 0).sum()}")
mw = stats.mannwhitneyu(F6[F6.group == "CA3"].coh_slope.dropna(), F6[F6.group == "CA1"].coh_slope.dropna(), alternative="greater")
print(f"CA3 slopes above CA1 slopes: Mann-Whitney p = {mw.pvalue:.3f}")
print("Mean coherence by segment:")
print(pd.DataFrame({g: np.nanmean(np.vstack(F6[F6.group == g].coh.values), 0) for g in ("CA3", "CA1")}).round(3).T.to_string())
print("Per subject, median slope:", F6.groupby(["group", "subject"]).coh_slope.median().round(4).to_dict())
print("Within animal, CA3 slopes above CA1 slopes (Mann-Whitney):")
for subj in ("DT5", "DT9"):
    g = F6[F6.subject == subj]
    a, b = g[g.group == "CA3"].coh_slope.dropna(), g[g.group == "CA1"].coh_slope.dropna()
    print(f"  {subj}: CA3 N = {len(a)}, median {a.median():.4f}; CA1 N = {len(b)}, median {b.median():.4f}; p = {stats.mannwhitneyu(a, b, alternative='greater').pvalue:.3f}")
F6v = F6all.assign(group=F6all.group.replace({"none": "CA1"}))
mwv = stats.mannwhitneyu(F6v[F6v.group == "CA3"].coh_slope.dropna(), F6v[F6v.group == "CA1"].coh_slope.dropna(), alternative="greater")
print(f"Variant, sessions without hippocampal units counted as CA1 (as before revision, N = {int((F6v.group == 'CA1').sum())}): CA3 above CA1, p = {mwv.pvalue:.3f}")

**Result: not reproduced.** Theta phase locking between the hippocampal and LS channels does not rise along the run in the 11 CA3 sessions (median slope 0.0026 per segment, signed-rank p = 0.83), and CA3 slopes are not above those of the 36 CA1 sessions (p = 0.35). Within DT5 (p = 0.18) and DT9 (p = 0.91) CA3 slopes are not above CA1 slopes either. The measure is a phase locking value, not the paper's multitaper coherence, and the hippocampal channel is chosen here, not the paper's pyramidal layer site.

## Theta-gamma coupling between hippocampus and LS (fig6d-cfc-stronger-ca3)

> Coupling between hippocampal theta phase and LS gamma amplitude increases along the run for both
> CA1 and CA3, and is stronger for CA3-LS than for CA1-LS.

In [ ]:
mw = stats.mannwhitneyu(F6[F6.group == "CA3"].cfc_mean.dropna(), F6[F6.group == "CA1"].cfc_mean.dropna(), alternative="greater")
print(f"Mean coupling (circular-linear r): CA3 median {F6[F6.group == 'CA3'].cfc_mean.median():.4f}, CA1 median {F6[F6.group == 'CA1'].cfc_mean.median():.4f}; "
      f"CA3 above CA1, Mann-Whitney p = {mw.pvalue:.3f}")
for g in ("CA3", "CA1"):
    x = F6[F6.group == g].cfc_slope.dropna()
    w = stats.wilcoxon(x)
    print(f"{g}: N = {len(x)}, median slope {x.median():.5f} per segment, signed-rank p = {w.pvalue:.3f}, positive in {(x > 0).sum()}")
print("Mean coupling by segment:")
print(pd.DataFrame({g: np.nanmean(np.vstack(F6[F6.group == g].cfc.values), 0) for g in ("CA3", "CA1")}).round(4).T.to_string())
print("Per subject, mean coupling:", F6.groupby(["group", "subject"]).cfc_mean.median().round(4).to_dict())
print("Within animal, CA3 coupling above CA1 coupling (Mann-Whitney):")
for subj in ("DT5", "DT9"):
    g = F6[F6.subject == subj]
    a, b = g[g.group == "CA3"].cfc_mean.dropna(), g[g.group == "CA1"].cfc_mean.dropna()
    print(f"  {subj}: CA3 N = {len(a)}, median {a.median():.4f}; CA1 N = {len(b)}, median {b.median():.4f}; p = {stats.mannwhitneyu(a, b, alternative='greater').pvalue:.3f}")
mwv = stats.mannwhitneyu(F6v[F6v.group == "CA3"].cfc_mean.dropna(), F6v[F6v.group == "CA1"].cfc_mean.dropna(), alternative="greater")
print(f"Variant, sessions without hippocampal units counted as CA1 (as before revision): CA3 above CA1, p = {mwv.pvalue:.3f}")

**Result: not reproduced.** Coupling between hippocampal theta phase and LS gamma amplitude rises along the run in both groups (signed-rank p = 0.002 in 11 CA3 sessions and 0.001 in 36 CA1 sessions), but it is not higher in CA3 than in CA1 sessions at p < 0.05 (medians 0.059 and 0.051, Mann-Whitney p = 0.089), which the criterion needs for a full or a partial outcome. Within DT5 CA3 coupling is higher (p = 0.011); within DT9 it is not (p = 0.88). Before revision, with the 19 sessions without hippocampal units counted as CA1, p was 0.041.

## Ensemble decoding (figs7-ensemble-ls-phase-best)

> The firing phases of ensembles of simultaneously recorded LS neurons read out position better than
> the firing phases or rates of an equal number of hippocampal neurons (p < 0.001 at 15 to 25 neurons).

In [ ]:
E = pd.DataFrame([dict(x, session=i, subject=results[i]["summary"]["subject"]) for i, r in enumerate(results) for x in r.get("ensemble", [])])
print("Conditions with at least 15 included LS and 15 hippocampal units:", E.drop_duplicates(["session", "condition"]).shape[0],
      "in", E.session.nunique(), "sessions;", E.drop_duplicates(["session", "condition"]).groupby("subject").size().to_dict())
for n, g in E.groupby("size"):
    v = {k: np.concatenate(g[k].values) for k in ("ls_phase", "ls_rate", "hpc_phase", "hpc_rate")}
    print(f"size {n}: {len(g)} conditions; mean MSE LS phase {v['ls_phase'].mean():.0f}, LS rate {v['ls_rate'].mean():.0f}, "
          f"HPC phase {v['hpc_phase'].mean():.0f}, HPC rate {v['hpc_rate'].mean():.0f}")
    for other in ("hpc_phase", "hpc_rate"):
        lower = stats.ttest_ind(v["ls_phase"], v[other], alternative="less")
        higher = stats.ttest_ind(v["ls_phase"], v[other], alternative="greater")
        print(f"    LS phase minus {other}: {v['ls_phase'].mean() - v[other].mean():+.0f}; one-sided p, LS phase lower = {lower.pvalue:.2g}, "
              f"LS phase higher = {higher.pvalue:.2g}")

**Result: not reproduced.** In 65 conditions from 13 sessions with at least 15 active LS and hippocampal units, LS phase ensembles of 15 units have a higher MSE (4,738) than hippocampal phase ensembles (4,597; one-sided p = 0.00058 for LS phase higher) and hippocampal rate ensembles (3,119; p = 3e-186). The same holds at 20 units (7 conditions). Only at 25 units (3 conditions) is the LS phase MSE lower than the hippocampal phase MSE (3,459 against 3,746, one-sided p = 0.0075), and it stays above the hippocampal rate MSE (2,259). That p value treats the ensemble draws within each of the 3 conditions as independent, so it overstates the evidence; at 15 and 20 units the direction is the reverse of the claim in any case.

## Hippocampal-LS assemblies, on a sample of pairs (fig5-assemblies)

> Spike trains of hippocampal neurons predicted LS firing in 18.5% (2,087 of 11,264) of
> hippocampal-LS pairs, with a median optimal timescale of 25 ms.

The peer prediction is run on a random sample of about 1% of the pairs, as all of them do not fit
the time limit (the estimate is printed below). The procedure is in `lshpc.py` and the claims file.

In [ ]:
AS = pd.DataFrame([dict(x, session=i, subject=results[i]["summary"]["subject"]) for i, r in enumerate(results) for x in r.get("assemblies", [])])
n_total = sum(r.get("assembly_pairs_total", 0) for r in results)
pairs_units = sum(len(set(LS[LS.session == s_].unit)) * len(set(HP[HP.session == s_].unit)) for s_ in S.session)
print(f"Pairs of active hippocampal and LS units recorded together: {pairs_units:,}; pairs active in the same condition, "
      f"counted per condition: {n_total:,} (paper 11,264 pairs)")
print(f"Sampled with p = {lshpc.ASSEMBLY_SAMPLE_P}: {len(AS)} pairs in {AS.session.nunique()} sessions; per subject {AS.groupby('subject').size().to_dict()}")
print("Fits without a finite strength:", int((~np.isfinite(AS.strength)).sum()))
is_a = AS.strength >= lshpc.ASSEMBLY_THRESHOLD
k, n = int(is_a.sum()), len(AS)
ci = stats.binomtest(k, n).proportion_ci(0.95)
print(f"Assemblies (strength at least 4): {k} of {n} = {k / n:.3f}, 95% interval {ci.low:.3f} to {ci.high:.3f} | paper 0.185")
print(f"Median best window of the assemblies: {AS[is_a].window_ms.median():.1f} ms (paper 25); of all sampled pairs {AS.window_ms.median():.1f} ms")
print("Best windows of the assemblies (ms: pairs):", AS[is_a].window_ms.value_counts().sort_index().to_dict())
print("Per subject, fraction of assemblies:", is_a.groupby(AS.subject).mean().round(3).to_dict())
for th in (1.5, 2, 4, 8):
    print(f"  threshold {th}: {(AS.strength >= th).mean():.3f} of sampled pairs")
print(f"Real gain above all 10 shuffles (strength above 1): {(AS.strength > 1).mean():.3f} of sampled pairs")
it = AS.strength_tuning >= lshpc.ASSEMBLY_THRESHOLD
ci2 = stats.binomtest(int(it.sum()), n).proportion_ci(0.95)
print(f"Variant, with the LS unit's rate map at the current position as a covariate: {int(it.sum())} of {n} = {it.mean():.3f} "
      f"(95% interval {ci2.low:.3f} to {ci2.high:.3f}); median best window {AS[it].window_ms_tuning.median():.1f} ms; "
      f"threshold 1.5: {(AS.strength_tuning >= 1.5).mean():.3f}; pairs found by both: {int((it & is_a).sum())}")
sec = AS.seconds.mean()
print(f"Time per pair: {sec:.2f} s on one process; all {n_total:,} pairs would take about {n_total * sec / 3600:.0f} process-hours")

**Result: not reproduced.** In a random sample of 670 of the 70,898 hippocampal-LS pairs active in the same condition, 35 (5.2%, 95% interval 3.7% to 7.2%) form an assembly of strength at least 4, against 18.5%, and the median best window of the assemblies is 20 ms, against 25 ms. The real gain beats all ten shuffles in 35.4% of the pairs. This cannot be compared with 1 in 11, since the window is chosen to maximise the real fit and the shuffles are run only at that window, so the real fit and the shuffles are not exchangeable without coupling. With a threshold of 1.5 the fraction is 19.4%. Position enters the fit as one linear term, so shared spatial tuning is not removed; adding the LS unit's rate map at the current position as a covariate gives 5.5% (95% interval 3.9% to 7.5%; 29 of the 35 assemblies found again), so the linear coding does not account for the low fraction. The bin size (1 ms), the Gaussian kernel and the window grid are choices made here, as the Methods do not give them and the paper's assembly code is not public. Most sampled pairs come from DT2 (408 of 670), and the pairs are counted per condition, as the Methods imply ("Each neuron pair has a single assembly strength value, determined by all spikes that occur during a single block of trials").

## Peak information of LS and hippocampal rate maps (ls-information-lower)

> LS rate maps have lower peak information rates than hippocampal place field maps (0.4 against
> 1.7 bits/cm; p < 1e-100).

In [ ]:
S["cm_per_bin"] = np.where(S.position_unit_used == "cm", S.path_length_median / lshpc.N_BINS, np.nan)
I = A.assign(cm_per_bin=S.cm_per_bin[A.session].values)
print("Included maps left out for positions in pixels (DT2):", int(I.cm_per_bin.isna().sum()))
I = I[I.cm_per_bin.notna()]
li, hi = I[I.region == "ls"], I[(I.region == "hpc") & (I.n_fields > 0)]
a, b = li.info_peak_bits / li.cm_per_bin, hi.info_peak_bits / hi.cm_per_bin
t = stats.ttest_ind(a, b)
print(f"LS rate maps: N = {len(a)}, mean {a.mean():.3f}, median {a.median():.3f} bits/cm (paper 0.4)")
print(f"Hippocampal place-field maps: N = {len(b)}, mean {b.mean():.3f}, median {b.median():.3f} bits/cm (paper 1.7)")
print(f"LS against hippocampus: t = {t.statistic:.1f}, p = {t.pvalue:.1e}")
print("Bin size, cm: median", round(float(I.drop_duplicates("session").cm_per_bin.median()), 2), "| range",
      round(float(I.cm_per_bin.min()), 2), "to", round(float(I.cm_per_bin.max()), 2))
print("Per subject, mean bits/cm:", pd.DataFrame({"LS": a.groupby(li.subject).mean(), "HPC fields": b.groupby(hi.subject).mean()}).round(3).to_dict())
ha = I[I.region == "hpc"]
print(f"Variant, every included hippocampal map: mean {(ha.info_peak_bits / ha.cm_per_bin).mean():.3f} bits/cm")
print(f"In bits per bin: LS mean {li.info_peak_bits.mean():.3f}, hippocampal fields {hi.info_peak_bits.mean():.3f} (at most log2(7) = {np.log2(7):.2f})")

**Result: partially reproduced.** LS rate maps carry less peak information than hippocampal place-field maps (means 0.632 and 1.490 bits/cm, t = -40.3, p = 6.0e-287, below the paper's 1e-100), but neither value is within tolerance of 0.4 and 1.7 under either reading (medians 0.556 and 1.435). The measure is that of fig1e (7 levels); the paper's normalization of it is not public, so the scale of these values is uncertain. DT2 is left out for its positions in pixels.

## Blocked claims

- fig5g-assembly-neurons-stronger-phase and fig6e-assembly-position need the assembly strength of
  every pair: which LS neurons take part in any assembly, and the strengths of all CA1 and CA3 place
  cells paired with LS. At about 0.6 s per pair on one process in this run (1.2 s in the previous one;
  printed under fig5-assemblies) the 70,898 pairs take about 12 process-hours, beyond the time limit,
  so both are blocked as not runnable. Sampling LS neurons and running all of their pairs (about 89
  per neuron) would test them on part of the data; that analysis was not run in this attempt.
- speed-independent: the paper does not define the rate of phase precession or its test of speed
  modulation, and no code for it is public. The data needed are in the dandiset.
- figs6-reference-frame: no file marks a maze rotation (see the data check).
- fig7-dynamic-weighting-model: a simulation that uses no recorded data.

## Summary

| Claim | Outcome |
|---|---|
| ls-neuron-count | Reproduced |
| hpc-unit-count | Partially reproduced |
| ls-active-neurons | Reproduced |
| fig2-ls-phase-vs-shuffle | Not reproduced |
| fig2-ls-phase-beats-rate | Not reproduced |
| fig2e-ls-phase-lower-mse | Reproduced |
| fig2fg-mse-difference | Partially reproduced |
| fig1i-hpc-rate-phase-equivalent | Not reproduced |
| ls-few-place-fields | Partially reproduced |
| hpc-place-field-properties | Partially reproduced |
| ls-place-fields-unstable | Not reproduced |
| fig1e-phase-info-smaller-scale | Partially reproduced |
| fig1g-phase-more-info-fine-scale | Partially reproduced |
| fig4d-depth-gradient | Partially reproduced |
| fig4e-onset-depth | Not reproduced |
| fig6c-ca3-coherence-increases | Not reproduced |
| fig6d-cfc-stronger-ca3 | Not reproduced |
| figs7-ensemble-ls-phase-best | Not reproduced |
| fig5-assemblies | Not reproduced (on a sample of pairs) |
| ls-information-lower | Partially reproduced |
| fig5g-assembly-neurons-stronger-phase, fig6e-assembly-position | Blocked: not runnable |
| speed-independent | Blocked: methods underspecified |
| figs6-reference-frame | Blocked: metadata insufficient |
| fig7-dynamic-weighting-model | Blocked: other |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.**

- The paper's linearized behaviour, trial conditions and choice of CA1 channel are not in the
  dandiset. Positions are linearized here by the fraction of each trial's path (200 bins, about 1.1
  cm), conditions are formed from trial type, direction and start and end sites, and the theta
  channel is the hippocampal channel with high theta and the most 100 to 250 Hz power. In the 11
  sessions with `ca3` channels it is a CA3 channel, and in DT5 its theta rhythm is weak.
- The per-cell decoder follows the paper's scratch code; the function that bins spike phases is not
  public, and the circular mean per bin is used.
- Place fields follow bz_getPlaceFields1D on the Gaussian rate map of bz_firingMap1D (SD 4 bins),
  with the Methods' limits converted to bins at 1.5 cm. This finds a third of the paper's share of
  hippocampal fields; without the code's rule dropping peaks whose SD across trials exceeds their
  mean it finds more than the paper in both regions (46.0% hippocampal, 20.0% LS).
- The information measure of Figure 1 and of the information rates is built from the Methods'
  formula with choices fixed in the claims file; the paper's function is not public.
- Coherence in Figure 6 is a phase locking value of the 4 to 12 Hz phases, not multitaper coherence.
- Only the trials are read from the LFP, with 1 s of padding for filtering.
- Assemblies (Figure 5) are computed on a random 1% of the pairs, with a bin size, kernel and
  window grid chosen here; the claims that need every pair (Figure 5G, 6E) were not tested.

**What was found about the dandiset.**

- The paper's processed behaviour (linearized position and trial conditions), the CA1 channel per
  session, and the unit recording sites are not in the dandiset. No session has both CA1 and CA3
  channels labelled, where the paper's Figure 6A has sessions with both.
- Eight DT8 files state positions in mm but hold metres; in every file the z column of position is a
  copy of y; eight of eleven DT2 files repeat or reverse position timestamps; DT2 positions are in
  pixels with no scale; the head-orientation quaternion is stated in metres.
- DT2 electrodes are all labelled `unknown`; DT7 20170317 has 43 units with no region label; DT7
  20170324 has no units; 11 units hold spike times out of order.
- The depths in the session names are not described; the first is taken as the LS depth.
- DT9 20170501's trials do not match its positions; DT8 20170207 and 20170224 have trials without
  type or direction.
- The units labelled `hpc` (2,593) are 12.9% more than the paper's 2,297 CA1 units.
- In the script that writes the paper's place-field files (scratch_07_25_17_calc_firingMaps.m),
  the file the figures load is named for a 20% threshold, but no threshold is passed, so the
  function's default of 1% applies.